# TP3: Windowed Aggregations & Visualization

**Goal:** Learn tumbling and sliding window aggregations on streaming or batch timestamped data, and export results for visualization.

This student notebook contains a runnable batch example and streaming skeleton. Use streaming if Kafka is available.

## Prerequisites
- If streaming: Kafka broker + spark-sql-kafka package.
- Python: pandas, matplotlib

## Quick help

### Introduction to Windows

In Spark Structured Streaming, a **window** allows you to group streaming events based on **time intervals**. This is useful to perform aggregations (like sums, counts, averages) over **fixed or sliding periods** of time.

#### Key concepts

- **Event time**: the timestamp of the event (not the processing time).  
- **Window duration**: the length of each window, e.g., 1 minute.  
- **Slide duration** (optional): how often a new window starts. If not specified, the windows do **not overlap** (tumbling windows).

#### Types of windows

1. **Tumbling window**: consecutive, non-overlapping windows.  
   - Example: 1-minute windows → `[00:00-00:01], [00:01-00:02], ...`

2. **Sliding window**: windows that can overlap.  
   - Example: 1-minute windows sliding every 30 seconds → `[00:00-00:01], [00:00:30-00:01:30], [00:01:00-00:02:00], ...`

#### How it works in Spark

```python
from pyspark.sql.functions import window

df.groupBy(
    window("timestamp", "1 minute", "30 seconds"),  # window duration and slide
    "category"
).count()
```

### Quick Spark / DataFrame reminder (cheat sheet)
- Create SparkSession: `from pyspark.sql import SparkSession; spark = SparkSession.builder.appName("app").getOrCreate()`
- Read CSV: `spark.read.option("header",True).csv("path")`
- Read Parquet: `spark.read.parquet("path")`
- Select columns: `df.select("col1","col2")`
- Filter rows: `df.filter(df.col > 10)` or `df.where("col > 10")`
- Add/modify column: `df.withColumn("new", expr(...))` or `df.withColumn("new", df.col * 2)`
- Cast column: `df.withColumn("ts", col("ts").cast("timestamp"))`
- Join: `df1.join(df2, on="key", how="left")`
- Aggregations: `df.groupBy("key").agg(count("*").alias("n"), sum("amount").alias("total"))`
- Cache/Persist: `df.cache()` then trigger with an action like `df.count()`
- Explain plan: `df.explain(True)` to see logical/physical plans
- Convert to pandas (small results): `df.toPandas()`
- Write Parquet: `df.write.mode("overwrite").parquet("out/")`
- For streaming: `spark.readStream.format("kafka")...` and `df.writeStream...start()`
- Use `checkpointLocation` for streaming durability


In [ ]:
# Cell 1: Batch mode example - prepare small events dataset with timestamps
import pandas as pd
from datetime import datetime, timedelta
import numpy as np

start = datetime(2024,1,1)
rows = []
for i in range(200):
    rows.append({'id':i, 'ts': start + timedelta(seconds=30*i), 'value': float((i%5)+1)})

ev = pd.DataFrame(rows)
ev.to_csv('data/events_batch.csv', index=False)
print('Wrote data/events_batch.csv')

In [ ]:
# Cell 2: Use Spark to perform windowed aggregation (batch example)
from pyspark.sql import SparkSession
from pyspark.sql.functions import window, col

spark = SparkSession.builder.appName('TP3_Window').getOrCreate()
df = spark.read.option('header',True).csv('data/events_batch.csv')
# convert ts to timestamp
from pyspark.sql.functions import to_timestamp

df2 = df.withColumn('event_time', to_timestamp(col('ts')))

agg = df2.groupBy(window(col('event_time'),'1 minute')).count().selectExpr('window.start as window_start','count')
res = agg.orderBy('window_start').toPandas()
res.to_csv('output/window_counts.csv', index=False)
print('Wrote output/window_counts.csv')
res.head()

## Streaming skeleton (if you want to use Kafka)
- Read from Kafka topic `events` (value is JSON with `event_time`), parse, and apply `groupBy(window(event_time, '1 minute')).count()`.
- For sliding windows, use `groupBy(window(event_time, '90 seconds','30 seconds'))`.

**Exercises:**
1. Run the batch example and plot counts versus time using matplotlib.
2. Implement tumbling (1 min) and sliding (90s window, slide 30s) aggregations in streaming mode (if Kafka available).

**Submission:** `output/window_counts.csv` and a plot image (PNG).